# 01 — Data Understanding & Descriptive Analysis

This notebook covers the **Data Understanding** phase of the CRISP-DM workflow, corresponding to `load_secom_data()`, `describe_secom_data()`, and `plot_section2_visualizations()` in `src/secom_analysis.py` and `src/plots.py`.

### Objectives
1. **Runtime Data Acquisition**: Download `secom.data` and `secom_labels.data` directly from the UCI Machine Learning Repository at runtime while keeping features (`X`) and target labels (`y`) strictly separate.
2. **Structural Inspection**: Verify matrix dimensions (`1567 × 590`), column data types (`float64`), and check for duplicate wafer rows.
3. **Target Variable & Class Imbalance**: Quantify Pass (`-1`) vs. Fail (`1`) wafer counts and percentages, and classify the imbalance tier.
4. **Feature Scale & Distribution Exploration**: Compute per-sensor descriptive statistics (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`) and inspect representative sensor histograms.
5. **Initial Missing-Value Analysis**: Quantify total missing cells, per-sensor missingness percentages, and average missing sensors per wafer across Pass vs. Fail classes.
6. **Decision & Impact Log (SIP Table)**: Record all Data Understanding decisions before splitting the dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

--- 
## Loading and Combining the SECOM Files

- **`secom.data`**: `1,567 × 590` matrix of anonymized semiconductor process sensor measurements. Missing values are encoded as literal `"NaN"` strings.
- **`secom_labels.data`**: `1,567` rows containing the binary quality test outcome (`-1` = Pass / good wafer, `1` = Fail / defective wafer) and test timestamp.
- **Why keep `X` and `y` separate?** Keeping features (`X`) and targets (`y`) as separate objects from the start prevents accidental target leakage during feature transformations.

In [ ]:
def load_secom_data():
    print("Downloading SECOM dataset via pandas...")
    base = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/"

    X = pd.read_csv(base + "secom.data", sep=" ", header=None)
    labels = pd.read_csv(
        base + "secom_labels.data", sep=" ", header=None, names=["target", "timestamp"]
    )

    y = labels["target"]
    X.columns = [f"sensor_{i}" for i in range(X.shape[1])]

    print(f"\nTotal wafers (rows): {len(X)}")
    print(f"Total labels (rows): {len(y)}")
    print(f"Total sensors (columns): {len(X.columns)}")
    return X, y


X, y = load_secom_data()

--- 
## Initial Structural Inspection & Target Class Distribution

- **Data types (`dtypes`)**: All 590 sensor columns parse as numeric (`float64`).
- **Duplicate wafer rows**: Checked via `X.duplicated().sum()` (`0` duplicate wafers).
- **Class Imbalance**:
  - **Pass (`-1`)**: `1,463` wafers (`93.36%`)
  - **Fail (`1`)**: `104` wafers (`6.64%`)
  - **Imbalance Tier**: Falls into the **moderate imbalance tier (`1% – 20%` minority class)** (~1:14 defect ratio). Because a naive classifier predicting "Pass" for every wafer would achieve `93.36%` accuracy while catching `0%` of defects, raw accuracy is misleading and downstream evaluation must prioritize **sensitivity (recall)**, **F1-score**, **ROC-AUC**, and **cost-based error weighting**.

In [ ]:
print("Column Data Types:")
print(X.dtypes.value_counts())
print(f"Duplicate wafer rows: {X.duplicated().sum()}")

class_counts = y.value_counts()
class_pct = (y.value_counts(normalize=True) * 100).round(2)
print("\nTarget Class Distribution (Pass = -1, Fail = 1):")
for label in [-1, 1]:
    print(f"  Class {label:2d}: {class_counts[label]} wafers ({class_pct[label]}%)")

--- 
## Exploring Feature Scales & Descriptive Statistics

Because SECOM sensors are anonymized (`sensor_0` … `sensor_589`), we rely on statistical summaries (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`) to inspect sensor behavior:
- **Widely differing scales**: Some sensors operate in small fractions (`0.001` – `1.0`), whereas others operate in the thousands (`1,000` – `30,000+`). Distance-based methods like kNN imputation will be dominated by large-range sensors unless features are scaled first.
- **Zero-variance (`std == 0`) sensors**: Several sensors (`sensor_5`, `sensor_13`, etc.) exhibit `std = 0.0` (`min == max`), indicating constant readings across all wafers.
- **Missingness (`count < 1567`)**: Non-null counts vary substantially across columns.

In [ ]:
feature_summary = X.describe().T[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
]
print("Sample Sensor Descriptive Statistics (First 10 Sensors):")
print(feature_summary.head(10).round(3).to_string())

Path("../results").mkdir(parents=True, exist_ok=True)
feature_summary.to_csv("../results/descriptive_statistics.csv")

--- 
## Initial Missing-Value Analysis (Overall, By Feature, and By Class)

We check three aspects of missingness before any splitting or removal:
1. **Overall missingness**: `41,951` missing cells out of `924,530` total cells (`4.54%`).
2. **By feature**: Missingness is heavily concentrated in a small subset of sensors (top sensors exceed `85%–91%` missing).
3. **By class (Pass vs. Fail check)**: Pass wafers average `26.95` missing sensors vs. `24.12` for Fail wafers on the full dataset — nearly identical, indicating that overall missingness is driven by random sensor/logging dropouts rather than wafer failure itself.

In [ ]:
total_missing = X.isna().sum().sum()
missing_pct_overall = (total_missing / X.size) * 100
missing_by_feature = (X.isna().mean() * 100).sort_values(ascending=False)
missing_by_class = X.isna().sum(axis=1).groupby(y).mean()

print(f"Total missing cells: {total_missing} ({missing_pct_overall:.2f}% of dataset)")
print("\nTop 10 Sensors by Missing Percentage (%):")
print(missing_by_feature.head(10).round(2))
print("\nAverage missing sensors per wafer by class (Full Dataset):")
print(f"Pass (-1): {missing_by_class[-1]:.2f} | Fail (1): {missing_by_class[1]:.2f}")

--- 
## Initial Visualisations

1. **Wafer Class Distribution (`class_distribution.png`)**: Visualizes the `93.36%` Pass vs. `6.64%` Fail imbalance.
2. **Sample Sensor Distributions (`sample_sensor_distributions.png`)**: Shows varying numeric scales and skewness across `sensor_0` – `sensor_3`.
3. **Missingness by Feature (`initial_missingness_by_feature.png`)**: Shows that a steep tail of ~24–32 sensors accounts for the extreme missingness rates (`>60%`), while the vast majority of sensors have `<5%` missing data.

In [ ]:
Path("../results/figures").mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Pass (-1)", "Fail (1)"], [class_counts[-1], class_counts[1]], color=["steelblue", "indianred"], edgecolor="black")
ax.set_title("Wafer Class Distribution (Pass vs. Fail)")
ax.set_ylabel("Number of Wafers")
fig.tight_layout()
plt.show()

sample_features = ["sensor_0", "sensor_1", "sensor_2", "sensor_3"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, col in zip(axes.flat, sample_features):
    ax.hist(X[col].dropna(), bins=30, edgecolor="black")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel("Sensor Reading")
    ax.set_ylabel("Frequency")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(missing_by_feature)), missing_by_feature.values, width=1.0)
ax.set_title("Missing Value Percentage by Sensor (Sorted)")
ax.set_xlabel("Sensors (Sorted by % Missing)")
ax.set_ylabel("Missing Values (%)")
fig.tight_layout()
plt.show()

--- 
## Decision & Impact Log (SIP Table — Data Understanding)

| Step / Decision Point | Options / Approaches Available | Pros & Cons of Each Method | Decision Made | Consequence (Before → After) |
| :--- | :--- | :--- | :--- | :--- |
| **Data Acquisition** | **Option A**: `ucimlrepo` wrapper (`fetch_ucirepo(id=179)`)<br>**Option B**: Direct runtime download via `pandas.read_csv()` from UCI URLs | **Option A**: Convenient wrapper, but prone to API/metadata timeouts returning `None`.<br>**Option B**: Zero external wrapper failure risk; reads space-delimited files directly into memory at runtime. | **Option B** (`pandas.read_csv` from UCI URLs) | `1,567` wafers and `590` sensors (`sensor_0`..`sensor_589`) loaded dynamically at runtime; `X` and `y` kept strictly separate. |
| **Target Coding** | **Option A**: Keep original UCI coding (`-1` = Pass, `1` = Fail)<br>**Option B**: Remap to `{0, 1}` | **Option A**: Matches UCI documentation and reference papers (`-1` vs `1`) directly.<br>**Option B**: Required only by specific loss functions expecting `{0, 1}`. | **Option A** (Preserve `-1` = Pass, `1` = Fail) | `1,463` Pass (`93.36%`) and `104` Fail (`6.64%`); no preprocessing applied prior to Train/Test split. |